In [ ]:
# CELL 1 - Runtime Information
from pathlib import Path
import sys,platform
from importlib.metadata import version
print(sys.version); print(platform.platform()); print(Path.cwd())
for p in ['pandas','numpy','matplotlib','scikit-learn','pillow','pytest']: print(p,version(p))


In [ ]:
# CELL 2 - Install Dependencies
import subprocess,sys
from pathlib import Path
Path('/content/requirements_stage8.txt').write_text('pandas>=2.2,<3.0\nnumpy>=1.26,<3.0\nmatplotlib>=3.8,<4.0\nscikit-learn>=1.4,<2.0\npillow>=10.0,<13.0\npytest>=8.0,<10.0\n')
subprocess.run([sys.executable,'-m','pip','install','-q','-r','/content/requirements_stage8.txt'],check=True)


In [ ]:
# CELL 3 - Create Project Directories
ROOT=Path('/content/DT25_Stage8_KMeans_Execution'); D={'src':ROOT/'src','tests':ROOT/'tests','input':ROOT/'data/input','processed':ROOT/'data/processed','tables':ROOT/'outputs/tables/clustering','figures':ROOT/'outputs/figures/clustering','logs':ROOT/'outputs/logs/stage8'}
for p in D.values(): p.mkdir(parents=True,exist_ok=True)


In [ ]:
# CELL 4 - Upload Input Files
from google.colab import files
import shutil
required={'rfm_customers.csv','rfm_standard_raw.csv','rfm_log_standard.csv','rfm_robust_raw.csv'}; uploaded=files.upload()
if set(uploaded)!=required: raise ValueError(f'Expected {sorted(required)}, received {sorted(uploaded)}')
for n in required: shutil.copy2(Path('/content')/n,D['input']/n)


In [ ]:
# CELL 5 - Validate Inputs
import sys,importlib
(D['src']/'cluster_evaluator.py').write_text('"""DT25 Stage 8 K-Means model-selection engine.\n\nEvaluates three accepted RFM preprocessing candidates across k=2..10,\nchecks multi-seed stability and cluster-size risks, selects exactly one\ncandidate-k pair using an explicit composite evidence rule, and exports\ntechnical ClusterID assignments. No PCA, ClusterName, synthetic data, or\nnon-K-Means model is created.\n"""\nfrom __future__ import annotations\nfrom dataclasses import dataclass\nfrom pathlib import Path\nfrom time import perf_counter\nfrom typing import Any\nimport hashlib, json\nimport numpy as np\nimport pandas as pd\nfrom sklearn.cluster import KMeans\nfrom sklearn.metrics import (silhouette_score, davies_bouldin_score,\n    calinski_harabasz_score, adjusted_rand_score)\n\nEXPECTED_CUSTOMERS=4338\nK_VALUES=list(range(2,11))\nSEEDS=[0,7,21,42,84,123,2026]\nPRIMARY_SEED=42\nN_INIT=20\nMAX_ITER=500\nTINY_THRESHOLD=0.01\nCANDIDATES={\n "STANDARD_RAW":"rfm_standard_raw.csv",\n "LOG_STANDARD":"rfm_log_standard.csv",\n "ROBUST_RAW":"rfm_robust_raw.csv",\n}\nFEATURES=["Recency_Scaled","Frequency_Scaled","Monetary_Scaled"]\n\nclass ClusterInputError(ValueError): pass\nclass ClusterAcceptanceError(RuntimeError): pass\n\n@dataclass(frozen=True)\nclass Paths:\n root: Path\n @property\n def input(self): return self.root/"data"/"input"\n @property\n def processed(self): return self.root/"data"/"processed"\n @property\n def tables(self): return self.root/"outputs"/"tables"/"clustering"\n @property\n def figures(self): return self.root/"outputs"/"figures"/"clustering"\n @property\n def logs(self): return self.root/"outputs"/"logs"/"stage8"\n def create(self):\n  for p in (self.processed,self.tables,self.figures,self.logs): p.mkdir(parents=True,exist_ok=True)\n\nclass ClusterEvaluator:\n def __init__(self,paths:Paths):\n  self.paths=paths; self.frames={}; self.hash_before={}; self.hash_after={}\n  self.rfm=None\n @staticmethod\n def sha256(path:Path)->str:\n  h=hashlib.sha256()\n  with path.open("rb") as f:\n   for b in iter(lambda:f.read(1024*1024),b""): h.update(b)\n  return h.hexdigest()\n def validate_inputs(self)->pd.DataFrame:\n  self.paths.create(); rows=[]\n  rpath=self.paths.input/"rfm_customers.csv"\n  if not rpath.exists() or rpath.stat().st_size<=0: raise ClusterInputError("Missing rfm_customers.csv")\n  self.hash_before[rpath.name]=self.sha256(rpath)\n  self.rfm=pd.read_csv(rpath,dtype={"CustomerID":"string"})\n  need={"CustomerID","Recency","Frequency","Monetary"}\n  if not need.issubset(self.rfm.columns) or len(self.rfm)!=EXPECTED_CUSTOMERS or not self.rfm.CustomerID.is_unique: raise ClusterInputError("Invalid rfm_customers.csv baseline/schema")\n  rows.append({"File":rpath.name,"Candidate":"RFM_REFERENCE","Rows":len(self.rfm),"Customers":self.rfm.CustomerID.nunique(),"Missing":int(self.rfm[list(need)].isna().sum().sum()),"Infinite":0,"SHA256":self.hash_before[rpath.name],"Status":"PASS"})\n  ids=self.rfm.CustomerID.reset_index(drop=True)\n  for name,file in CANDIDATES.items():\n   p=self.paths.input/file\n   if not p.exists() or p.stat().st_size<=0: raise ClusterInputError(f"Missing {file}")\n   self.hash_before[file]=self.sha256(p)\n   df=pd.read_csv(p,dtype={"CustomerID":"string"})\n   missing=sorted(set(["CustomerID"]+FEATURES)-set(df.columns))\n   numeric=df[FEATURES] if not missing else pd.DataFrame()\n   if missing or len(df)!=EXPECTED_CUSTOMERS or not df.CustomerID.is_unique: raise ClusterInputError(f"Invalid {file}: {missing}")\n   if not df.CustomerID.reset_index(drop=True).equals(ids): raise ClusterInputError(f"Customer order/set mismatch: {file}")\n   if numeric.isna().any().any() or not np.isfinite(numeric.to_numpy()).all(): raise ClusterInputError(f"NaN/inf: {file}")\n   self.frames[name]=df\n   rows.append({"File":file,"Candidate":name,"Rows":len(df),"Customers":df.CustomerID.nunique(),"Missing":0,"Infinite":0,"SHA256":self.hash_before[file],"Status":"PASS"})\n  return pd.DataFrame(rows)\n def fit_candidate(self,name:str,k:int,seed:int=PRIMARY_SEED)->dict[str,Any]:\n  x=self.frames[name][FEATURES].to_numpy(); start=perf_counter()\n  model=KMeans(n_clusters=k,init="k-means++",n_init=N_INIT,max_iter=MAX_ITER,random_state=seed,algorithm="lloyd")\n  labels=model.fit_predict(x); runtime=perf_counter()-start\n  sizes=np.bincount(labels,minlength=k); min_size=int(sizes.min()); max_size=int(sizes.max())\n  return {"Candidate":name,"K":k,"Seed":seed,"Inertia":float(model.inertia_),"Silhouette":float(silhouette_score(x,labels)),"DaviesBouldin":float(davies_bouldin_score(x,labels)),"CalinskiHarabasz":float(calinski_harabasz_score(x,labels)),"Iterations":int(model.n_iter_),"RuntimeSeconds":runtime,"MinClusterSize":min_size,"MaxClusterSize":max_size,"MinClusterRatio":min_size/len(x),"MaxClusterRatio":max_size/len(x),"ImbalanceRatio":max_size/min_size,"ClusterSizes":json.dumps(sizes.tolist()),"Labels":labels,"Model":model}\n def evaluate_k_range(self)->pd.DataFrame:\n  records=[]\n  for name in CANDIDATES:\n   for k in K_VALUES:\n    d=self.fit_candidate(name,k); records.append({x:y for x,y in d.items() if x not in {"Labels","Model"}})\n  return pd.DataFrame(records)\n @staticmethod\n def detect_elbow(metrics:pd.DataFrame)->pd.DataFrame:\n  rows=[]\n  for name,g in metrics.groupby("Candidate"):\n   g=g.sort_values("K"); y=g.Inertia.to_numpy(); curvature=np.full(len(y),np.nan)\n   if len(y)>=3: curvature[1:-1]=y[:-2]-2*y[1:-1]+y[2:]\n   for (_,r),c in zip(g.iterrows(),curvature): rows.append({"Candidate":name,"K":int(r.K),"Inertia":r.Inertia,"DiscreteCurvature":c,"ElbowCandidate":bool(np.isfinite(c) and c==np.nanmax(curvature))})\n  return pd.DataFrame(rows)\n @staticmethod\n def shortlist(metrics:pd.DataFrame)->pd.DataFrame:\n  d=metrics.copy(); d["TinyWarning"]=d.MinClusterRatio<TINY_THRESHOLD\n  for c,asc in [("Silhouette",False),("DaviesBouldin",True),("CalinskiHarabasz",False),("ImbalanceRatio",True)]: d[c+"Rank"]=d.groupby("Candidate")[c].rank(ascending=asc,method="min")\n  d["WithinCandidateComposite"]=d[["SilhouetteRank","DaviesBouldinRank","CalinskiHarabaszRank","ImbalanceRatioRank"]].mean(axis=1)+d.TinyWarning.astype(int)*3\n  d["Shortlisted"]=False\n  for name,g in d.groupby("Candidate"): d.loc[g.nsmallest(2,"WithinCandidateComposite").index,"Shortlisted"]=True\n  return d\n def evaluate_multi_seed_stability(self,shortlisted:pd.DataFrame)->tuple[pd.DataFrame,pd.DataFrame,pd.DataFrame]:\n  runs=[]; ari_rows=[]; summaries=[]\n  for _,row in shortlisted.loc[shortlisted.Shortlisted].iterrows():\n   labels={}\n   for seed in SEEDS:\n    d=self.fit_candidate(row.Candidate,int(row.K),seed); labels[seed]=d["Labels"]\n    runs.append({x:y for x,y in d.items() if x not in {"Labels","Model"}})\n   matrix=pd.DataFrame(index=SEEDS,columns=SEEDS,dtype=float)\n   for a in SEEDS:\n    for b in SEEDS: matrix.loc[a,b]=adjusted_rand_score(labels[a],labels[b]); ari_rows.append({"Candidate":row.Candidate,"K":int(row.K),"SeedA":a,"SeedB":b,"ARI":matrix.loc[a,b]})\n   vals=[matrix.loc[a,b] for i,a in enumerate(SEEDS) for b in SEEDS[i+1:]]\n   summaries.append({"Candidate":row.Candidate,"K":int(row.K),"MeanARI":float(np.mean(vals)),"StdARI":float(np.std(vals)),"MinARI":float(np.min(vals)),"MaxARI":float(np.max(vals)),"StabilityStatus":"PASS" if np.mean(vals)>=0.80 and np.min(vals)>=0.60 else "WARNING"})\n  return pd.DataFrame(runs),pd.DataFrame(ari_rows),pd.DataFrame(summaries)\n def build_selection_matrix(self,shortlisted,stability)->pd.DataFrame:\n  d=shortlisted.merge(stability,on=["Candidate","K"],how="left"); d["WarningFlags"]=np.where(d.TinyWarning,"TINY_CLUSTER",""); d["InterpretabilityNote"]="Technical RFM-space partition; marketing names deferred"\n  sub=d.loc[d.Shortlisted].copy()\n  for c,asc in [("Silhouette",False),("DaviesBouldin",True),("CalinskiHarabasz",False),("ImbalanceRatio",True),("MeanARI",False)]: sub[c+"GlobalRank"]=sub[c].rank(ascending=asc,method="min")\n  sub["SelectionScore"]=sub[[c+"GlobalRank" for c in ["Silhouette","DaviesBouldin","CalinskiHarabasz","ImbalanceRatio","MeanARI"]]].mean(axis=1)+sub.TinyWarning.astype(int)*4+sub.StabilityStatus.ne("PASS").astype(int)*3\n  selected=sub.sort_values(["SelectionScore","Silhouette"],ascending=[True,False]).index[0]\n  d["SelectionStatus"]="REJECTED"; d.loc[d.Shortlisted,"SelectionStatus"]="SHORTLISTED"; d.loc[selected,"SelectionStatus"]="SELECTED"\n  if d.SelectionStatus.eq("SELECTED").sum()!=1: raise ClusterAcceptanceError("Selection must be unique")\n  return d\n def tiny_cluster_review(self,selection:pd.DataFrame)->pd.DataFrame:\n  rows=[]\n  for _,r in selection.loc[selection.SelectionStatus.isin(["SHORTLISTED","SELECTED"])].iterrows():\n   d=self.fit_candidate(r.Candidate,int(r.K)); labels=d["Labels"]; raw=self.rfm.copy(); raw["ClusterID"]=labels\n   for cid,g in raw.groupby("ClusterID"):\n    rows.append({"Candidate":r.Candidate,"K":int(r.K),"ClusterID":int(cid),"Customers":len(g),"Ratio":len(g)/len(raw),"MedianRecency":g.Recency.median(),"MedianFrequency":g.Frequency.median(),"MedianMonetary":g.Monetary.median(),"MaxFrequency":g.Frequency.max(),"MaxMonetary":g.Monetary.max(),"TinyWarning":len(g)/len(raw)<TINY_THRESHOLD,"Review":"WARNING_REVIEW_EXTREMES_NO_AUTOMATIC_REMOVAL" if len(g)/len(raw)<TINY_THRESHOLD else "PASS"})\n  return pd.DataFrame(rows)\n def export_final(self,selection:pd.DataFrame)->tuple[pd.DataFrame,pd.DataFrame]:\n  s=selection.loc[selection.SelectionStatus.eq("SELECTED")].iloc[0]; d=self.fit_candidate(s.Candidate,int(s.K),PRIMARY_SEED)\n  out=self.rfm[["CustomerID","Recency","Frequency","Monetary"]].copy(); out["ClusterID"]=d["Labels"]; out["PreprocessingCandidate"]=s.Candidate; out["K"]=int(s.K); out["RandomState"]=PRIMARY_SEED; out["NInit"]=N_INIT; out["MaxIter"]=MAX_ITER\n  cfg=pd.DataFrame([{"SelectedPreprocessing":s.Candidate,"SelectedK":int(s.K),"Init":"k-means++","NInit":N_INIT,"MaxIter":MAX_ITER,"RandomState":PRIMARY_SEED,"Algorithm":"lloyd","Silhouette":d["Silhouette"],"DaviesBouldin":d["DaviesBouldin"],"CalinskiHarabasz":d["CalinskiHarabasz"],"Inertia":d["Inertia"],"Iterations":d["Iterations"],"RuntimeSeconds":d["RuntimeSeconds"],"ClusterNamesCreated":"NO","PCACreated":"NO"}])\n  return out,cfg\n def verify_input_integrity(self)->pd.DataFrame:\n  rows=[]\n  for file in ["rfm_customers.csv",*CANDIDATES.values()]:\n   p=self.paths.input/file; self.hash_after[file]=self.sha256(p); rows.append({"File":file,"SHA256Before":self.hash_before[file],"SHA256After":self.hash_after[file],"Unchanged":self.hash_before[file]==self.hash_after[file],"SizeBytes":p.stat().st_size})\n  d=pd.DataFrame(rows)\n  if not d.Unchanged.all(): raise ClusterAcceptanceError("Input changed")\n  return d\n')
if str(D['src']) not in sys.path: sys.path.insert(0,str(D['src']))
from cluster_evaluator import ClusterEvaluator,Paths,K_VALUES,SEEDS,TINY_THRESHOLD
engine=ClusterEvaluator(Paths(ROOT)); input_verification=engine.validate_inputs(); input_verification.to_csv(D['tables']/'input_verification.csv',index=False); display(input_verification)


In [ ]:
# CELL 6 - Input SHA-256 Baseline
print(input_verification[['File','SHA256']].to_string(index=False))


In [ ]:
# CELL 7 - Write or Import cluster_evaluator.py
print('Module loaded:',D['src']/'cluster_evaluator.py')


In [ ]:
# CELL 8 - Load Three Candidates
for name,df in engine.frames.items(): print(name,df.shape,df.CustomerID.nunique())


In [ ]:
# CELL 9 - Evaluate k Range
metrics=engine.evaluate_k_range(); metrics.to_csv(D['tables']/'kmeans_metrics_all.csv',index=False); display(metrics)


In [ ]:
# CELL 10 - Create Elbow Analysis
import matplotlib.pyplot as plt
elbow=engine.detect_elbow(metrics); elbow.to_csv(D['tables']/'elbow_candidates.csv',index=False)
for name,g in metrics.groupby('Candidate'):
 fig,ax=plt.subplots(figsize=(8,5)); ax.plot(g.K,g.Inertia,marker='o'); ax.set_title(f'Elbow curve - {name}'); ax.set_xlabel('k'); ax.set_ylabel('Inertia'); fig.tight_layout(); fig.savefig(D['figures']/f"elbow_{name.lower()}.png",dpi=160); plt.close(fig)


In [ ]:
# CELL 11 - Create Metric Comparison
for col,title,file,ylabel in [('Silhouette','Silhouette theo k','silhouette_comparison.png','Silhouette (cao hơn tốt hơn)'),('DaviesBouldin','Davies-Bouldin theo k','davies_bouldin_comparison.png','Davies-Bouldin (thấp hơn tốt hơn)'),('CalinskiHarabasz','Calinski-Harabasz theo k','calinski_harabasz_comparison.png','Calinski-Harabasz (cao hơn tốt hơn)')]:
 fig,ax=plt.subplots(figsize=(9,5))
 for name,g in metrics.groupby('Candidate'): ax.plot(g.K,g[col],marker='o',label=name)
 ax.set_title(title); ax.set_xlabel('k'); ax.set_ylabel(ylabel); ax.legend(); fig.tight_layout(); fig.savefig(D['figures']/file,dpi=160); plt.close(fig)


In [ ]:
# CELL 12 - Shortlist Candidate-k Pairs
shortlisted=engine.shortlist(metrics); display(shortlisted.loc[shortlisted.Shortlisted])


In [ ]:
# CELL 13 - Multi-Seed Stability
multi_seed,ari,stability=engine.evaluate_multi_seed_stability(shortlisted); multi_seed.to_csv(D['tables']/'multi_seed_stability.csv',index=False); display(stability)


In [ ]:
# CELL 14 - ARI Stability
ari.to_csv(D['tables']/'ari_stability_matrix.csv',index=False)
fig,ax=plt.subplots(figsize=(10,5)); stability.assign(Model=stability.Candidate+' k='+stability.K.astype(str)).plot.bar(x='Model',y='MeanARI',ax=ax,legend=False); ax.set_title('Độ ổn định ARI của shortlist'); ax.set_ylabel('Mean ARI'); fig.tight_layout(); fig.savefig(D['figures']/'stability_comparison.png',dpi=160); plt.close(fig)


In [ ]:
# CELL 15 - Tiny Cluster Review
temp=engine.build_selection_matrix(shortlisted,stability); tiny=engine.tiny_cluster_review(temp); tiny.to_csv(D['tables']/'tiny_cluster_review.csv',index=False); tiny.to_csv(D['tables']/'cluster_size_analysis.csv',index=False)
fig,ax=plt.subplots(figsize=(11,5)); tiny.assign(Model=tiny.Candidate+' k='+tiny.K.astype(str)).pivot_table(index='ClusterID',columns='Model',values='Customers').plot.bar(ax=ax); ax.set_title('So sánh kích thước cụm'); ax.set_ylabel('Khách hàng'); fig.tight_layout(); fig.savefig(D['figures']/'cluster_size_comparison.png',dpi=160); plt.close(fig)


In [ ]:
# CELL 16 - Build Selection Matrix
selection=temp; selection.to_csv(D['tables']/'candidate_selection_matrix.csv',index=False); display(selection.sort_values(['SelectionStatus','SelectionScore']))


In [ ]:
# CELL 17 - Select Final Candidate and k
selected=selection.loc[selection.SelectionStatus.eq('SELECTED')]
if len(selected)!=1: raise AssertionError('Exactly one model must be selected')
decision=selected.copy(); decision['DecisionLogic']='Composite metric ranks + stability + size/outlier warnings; no single metric dominates'; decision.to_csv(D['tables']/'model_selection_decision.csv',index=False); display(decision)


In [ ]:
# CELL 18 - Fit Final K-Means
assignments,config=engine.export_final(selection); config.to_csv(D['tables']/'final_model_configuration.csv',index=False); display(config)


In [ ]:
# CELL 19 - Export Customer Assignments
assignments.to_csv(D['processed']/'kmeans_customer_assignments.csv',index=False); print(assignments.shape,assignments.ClusterID.nunique())


In [ ]:
# CELL 20 - Output Read-Back
# Export acceptance after basic invariants
checks=[('S8-01','Three candidates',len(engine.frames)==3),('S8-02','Full k grid',len(metrics)==27),('S8-03','Unique selected model',len(selected)==1),('S8-04','Assignment rows',len(assignments)==4338),('S8-05','No empty clusters',assignments.ClusterID.nunique()==int(config.SelectedK.iloc[0])),('S8-06','Stability exists',not stability.empty),('S8-07','No ClusterName/PCA',not any('ClusterName' in c for c in assignments.columns))]
acceptance=pd.DataFrame([{'CheckID':i,'Condition':x,'Status':'PASS' if ok else 'FAIL'} for i,x,ok in checks]); acceptance.to_csv(D['tables']/'stage8_acceptance_matrix.csv',index=False)
paths=list(D['tables'].glob('*.csv'))+[D['processed']/'kmeans_customer_assignments.csv']; rows=[]
for p in paths:
 df=pd.read_csv(p,low_memory=False); rows.append({'Path':str(p.relative_to(ROOT)),'Exists':True,'SizeBytes':p.stat().st_size,'Readable':True,'Rows':len(df),'Status':'PASS' if not df.empty else 'FAIL'})
out=pd.DataFrame(rows); out.to_csv(D['tables']/'output_verification.csv',index=False); display(out)


In [ ]:
# CELL 21 - Chart Validation
from PIL import Image
rows=[]
for p in sorted(D['figures'].glob('*.png')):
 with Image.open(p) as im: w,h=im.size
 rows.append({'Path':str(p.relative_to(ROOT)),'SizeBytes':p.stat().st_size,'Width':w,'Height':h,'Status':'PASS' if p.stat().st_size>0 and w>=400 and h>=300 else 'FAIL'})
cv=pd.DataFrame(rows); cv.to_csv(D['tables']/'chart_validation.csv',index=False); display(cv)


In [ ]:
# CELL 22 - Input SHA-256 After Execution
checksum=engine.verify_input_integrity(); checksum.to_csv(D['tables']/'input_checksum_report.csv',index=False); display(checksum)


In [ ]:
# CELL 23 - Automated Acceptance Tests
import subprocess,os
(D['tests']/'test_stage8_acceptance.py').write_text("from pathlib import Path\nimport numpy as np, pandas as pd\nfrom PIL import Image\nROOT=Path(__file__).resolve().parents[1]; T=ROOT/'outputs/tables/clustering'; F=ROOT/'outputs/figures/clustering'; P=ROOT/'data/processed'\n\ndef test_inputs_candidates():\n v=pd.read_csv(T/'input_verification.csv'); assert set(v.Candidate)=={'RFM_REFERENCE','STANDARD_RAW','LOG_STANDARD','ROBUST_RAW'}; assert v.Status.eq('PASS').all(); assert (v.Rows==4338).all()\ndef test_k_grid_and_metrics():\n m=pd.read_csv(T/'kmeans_metrics_all.csv'); assert len(m)==27; assert set(m.K)==set(range(2,11)); assert set(m.Candidate)=={'STANDARD_RAW','LOG_STANDARD','ROBUST_RAW'}; assert m[['Inertia','Silhouette','DaviesBouldin','CalinskiHarabasz']].notna().all().all()\ndef test_unique_selection_and_assignments():\n s=pd.read_csv(T/'candidate_selection_matrix.csv'); assert s.SelectionStatus.eq('SELECTED').sum()==1; row=s.loc[s.SelectionStatus.eq('SELECTED')].iloc[0]; assert int(row.K) in range(2,11)\n a=pd.read_csv(P/'kmeans_customer_assignments.csv',dtype={'CustomerID':'string'}); assert len(a)==4338 and a.CustomerID.is_unique; assert a.ClusterID.nunique()==int(row.K); assert not any('ClusterName' in c for c in a.columns)\ndef test_stability_and_ari():\n st=pd.read_csv(T/'multi_seed_stability.csv'); ari=pd.read_csv(T/'ari_stability_matrix.csv'); assert set(st.Seed)=={0,7,21,42,84,123,2026}; assert ari.ARI.between(-1,1).all()\ndef test_integrity_outputs_charts():\n c=pd.read_csv(T/'input_checksum_report.csv'); assert c.Unchanged.all(); o=pd.read_csv(T/'output_verification.csv'); assert o.Status.eq('PASS').all(); cv=pd.read_csv(T/'chart_validation.csv'); assert cv.Status.eq('PASS').all()\n for p in F.glob('*.png'):\n  with Image.open(p) as im: assert im.width>=400 and im.height>=300\ndef test_acceptance_and_prohibitions():\n a=pd.read_csv(T/'stage8_acceptance_matrix.csv'); assert a.Status.eq('PASS').all(); names=[p.name.lower() for p in ROOT.rglob('*') if p.is_file()]; assert not any('pca' in n or 'clustername' in n for n in names)\n")
r=subprocess.run([sys.executable,'-m','pytest','-q',str(D['tests']/'test_stage8_acceptance.py')],cwd=ROOT,text=True,capture_output=True); pytest_text=r.stdout+'\n'+r.stderr; (D['logs']/'pytest_output.txt').write_text(pytest_text); print(pytest_text)
if r.returncode!=0: raise AssertionError('Pytest failed')


In [ ]:
# CELL 24 - Final Machine-Readable Acceptance Summary
from datetime import datetime,timezone
s=config.iloc[0]; tiny_status='WARNING' if tiny.TinyWarning.any() else 'PASS'; failed=[]
if r.returncode!=0: failed.append('PYTEST')
if not out.Status.eq('PASS').all(): failed.append('OUTPUT_READBACK')
if not cv.Status.eq('PASS').all(): failed.append('CHARTS')
if not checksum.Unchanged.all(): failed.append('INPUT_INTEGRITY')
status='PASS' if not failed else 'FAIL'
summary='\n'.join([f'STAGE_8_EXECUTION_STATUS = {status}',f'STAGE_8_ACCEPTANCE_CANDIDATE = {status}','',f'INPUT_CUSTOMERS = 4338','PREPROCESSING_CANDIDATES = 3/3 VERIFIED',f'K_RANGE = {K_VALUES}',f'CANDIDATE_K_COMBINATIONS_EVALUATED = {len(metrics)}',f'SHORTLISTED_MODELS = {int(shortlisted.Shortlisted.sum())}',f'SELECTED_PREPROCESSING = {s.SelectedPreprocessing}',f'SELECTED_K = {int(s.SelectedK)}',f'SELECTED_SILHOUETTE = {s.Silhouette}',f'SELECTED_DAVIES_BOULDIN = {s.DaviesBouldin}',f'SELECTED_CALINSKI_HARABASZ = {s.CalinskiHarabasz}',f'SELECTED_INERTIA = {s.Inertia}',f'MULTI_SEED_RUNS = {len(SEEDS)}',f'STABILITY_STATUS = {selected.StabilityStatus.iloc[0]}',f'TINY_CLUSTER_REVIEW = {tiny_status}',f'FINAL_ASSIGNMENT_ROWS = {len(assignments)}',f'FINAL_CLUSTER_COUNT = {assignments.ClusterID.nunique()}','EMPTY_CLUSTERS = 0',f'OUTPUT_READ_BACK = {"PASS" if out.Status.eq("PASS").all() else "FAIL"}',f'CHART_VALIDATION = {"PASS" if cv.Status.eq("PASS").all() else "FAIL"}',f'PYTEST_STATUS = {"PASS" if r.returncode==0 else "FAIL"}',f'INPUT_FILES_UNCHANGED = {"YES" if checksum.Unchanged.all() else "NO"}','KMEANS_EXECUTED = YES','K_SELECTED = YES','CLUSTER_NAMES_CREATED = NO','PCA_CREATED = NO','OPEN_CONDITIONS = ["Final Chat evidence review pending", "Peer review evidence remains pending", "SQLite second-format confirmation remains open"]',f'FAILED_CHECKS = {failed if failed else "NONE"}','NOT_VERIFIED_ITEMS = NONE'])
(D['logs']/'acceptance_summary.txt').write_text(summary); (D['logs']/'execution_log.txt').write_text(f'ENTRY_POINT=ClusterEvaluator Stage8\nSTATUS={status}\nFINISHED_UTC={datetime.now(timezone.utc).isoformat()}\nEXCEPTION=NONE\n'); (D['logs']/'environment_info.txt').write_text(f'Python={sys.version}\nPlatform={platform.platform()}\n'); print(summary)


In [ ]:
# CELL 25 - Build and Download Evidence ZIP
import zipfile
from google.colab import files
zip_path=ROOT/'DT25_Stage8_KMeans_Evidence.zip'; evidence=[p for p in ROOT.rglob('*') if p.is_file() and D['input'] not in p.parents and p!=zip_path]
with zipfile.ZipFile(zip_path,'w',zipfile.ZIP_DEFLATED) as z:
 for p in evidence: z.write(p,p.relative_to(ROOT))
print(zip_path,zip_path.stat().st_size); files.download(str(zip_path))
